# VMBench workshop

Components: **PAS** amplitude · **OIS** human structure · **TCS** coherence · **MSS** smoothness · **CAS** learned plausibility.


## 0. Initialization

Validate the existing submodule and GPU, prepare dependencies, and download selected checkpoints. MSS defaults to instructor scores on T4.


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "utils.py").is_file():
    PROJECT_DIR = Path("/content/workshop")
if not (PROJECT_DIR / "utils.py").is_file():
    raise RuntimeError("Workshop checkout is missing. Prepare /content/workshop with vendor/vmbench first.")
sys.path.insert(0, str(PROJECT_DIR))
from utils import initialize, compute_pas, compute_ois, compute_tcs, compute_mss, compute_cas, compare_variants

METRICS_TO_RUN = ["PAS", "OIS", "TCS", "MSS", "CAS"]
VIDEO_META = {}  # filename -> {"subject_noun": "person", "subject_type": "human", "prompt": "..."}
CREATE_VARIANTS = False
workshop = initialize(PROJECT_DIR, metrics=METRICS_TO_RUN, video_meta=VIDEO_META,
                      mss_mode="precomputed", precomputed_url="", create_variants=CREATE_VARIANTS)


## 1. Videos

Use the local `vids/` folder or upload MP4/MOV/WEBM. Prepared files keep their original names and bytes. Set `VIDEO_META` for non-human subjects.


In [ ]:
VIDEO_DIR = PROJECT_DIR / "vids"
input_paths = workshop.load_videos(folder=VIDEO_DIR, limit=4)
video_info_df = workshop.preview_videos()
VIDEO_PATH, VIDEO_PATHS = workshop.select_videos()


## 2. Speed experiment (optional)

Enable `CREATE_VARIANTS` above for fast ×2, slow ×0.5, frame drop and duplication. Predict the scores before running.


In [ ]:
if CREATE_VARIANTS:
    VARIANTS = workshop.prepare_variants()
    VIDEO_PATH, VIDEO_PATHS = workshop.select_videos()


## PAS

Amplitude (raw displacement). Subject tracking with camera-motion compensation.


In [ ]:
if "PAS" in METRICS_TO_RUN:
    pas = compute_pas(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("PAS")


## OIS

Human structural integrity. N/A when a person or pose cannot be detected.


In [ ]:
if "OIS" in METRICS_TO_RUN:
    ois = compute_ois(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("OIS")


## TCS

Object persistence over time. Check detection warnings when interpreting high scores.


In [ ]:
if "TCS" in METRICS_TO_RUN:
    tcs = compute_tcs(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("TCS")


## MSS

Temporal artifacts. Live MSS also needs PAS; instructor CSV is the default on T4.


In [ ]:
if "MSS" in METRICS_TO_RUN:
    mss = compute_mss(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("MSS")


## CAS

Learned plausibility, not a physics simulator.


In [ ]:
if "CAS" in METRICS_TO_RUN:
    cas = compute_cas(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("CAS")


## Results

Compare each metric separately. Missing scores are not zeros; use diagnostics for status, source and runtime.


In [ ]:
workshop.show_metric_summary(VIDEO_PATH)
results_df = workshop.evaluate_dataset()
workshop.plot_results(results_df)
# Download in Colab:
# from google.colab import files
# files.download("/content/vmbench_results.csv")


## Discuss

- Which metric reacted to the speed change?
- Where did a score disagree with your visual judgment?
- Are these scores enough to distinguish real and generated videos?
